# Visualize static PDBs from protein databank

<video controls src="./assets/pdb_visualization.webm">

Fetch PDBs and convert to MDAnalysis universes:

In [1]:
from fetch_pdbs import fetch_pdb_universes

universes = fetch_pdb_universes("1AKE", "4BWZ", "3M1N", to_guess=("bonds", "types", "masses"))

Convert to NanoVer simulations:

In [2]:
from nanover.mdanalysis import UniverseSimulation

simulations = [UniverseSimulation.from_universe(universe, name=name) for name, universe in universes.items()]

Setup runner with simulations:

In [3]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(*simulations, port=0, name="EXAMPLE: Visualise static PDBs from databank")
imd_runner.load(0)

Use Nanover jupyter utilities to make convenience commands for switching between simulations:

In [4]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [5]:
def make_switch(i, simulation):
    def switch():
        imd_runner.load(i)
        utilities.notify_all(f"Loading {simulation.name}")
    return switch

for i, simulation in enumerate(simulations):
    imd_runner.app_server.register_command(f"user/sim-{i}", make_switch(i, simulation), label=simulation.name, icon="🫧")